# Day 2 Lab: The Error-State Extended Kalman Filter (ES-EKF)
**Instructor Edition (Solutions Included)**

---
### 🎯 Learning Objectives
1. Understand the fundamental separation between the **Nominal State** $\hat{\mathbf{x}}$ and the **Error State** $\delta\mathbf{x}$.
2. Learn why ES-EKF is the industry standard for aerospace, GNSS/INS sensor fusion, and autonomous vehicles.
3. Formulate the high-rate kinematic state integration and low-rate error-state Kalman update.
4. Understand continuous white noise spectral density scaling for $\mathbf{Q}_{\text{imu}} = \mathbf{Q}_c \Delta t$ and GNSS $\mathbf{R}_{\text{gps}}$.
5. Implement error injection $(\hat{\mathbf{x}} \leftarrow \check{\mathbf{x}} \oplus \delta\hat{\mathbf{x}})$ and error reset $(\delta\hat{\mathbf{x}} \leftarrow \mathbf{0})$.
6. Simulate GNSS/IMU dead-reckoning with periodic GPS outages and observe uncertainty growth and recovery via Plotly.


## 1. Environment Setup


In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import chi2

np.random.seed(42)
print('Environment ready: NumPy, SciPy, and Plotly loaded.')


---
## 2. Theory: Nominal vs. Error State Formulation

$$\mathbf{x} = \hat{\mathbf{x}} \oplus \delta\mathbf{x}$$

1. **High-rate Nominal Propagation:** $\check{\mathbf{x}}_k = \mathbf{f}(\hat{\mathbf{x}}_{k-1}, \mathbf{u}_{k-1})$
2. **Error Covariance Propagation:** $\check{\mathbf{P}}_k = \mathbf{F}_{k-1}\hat{\mathbf{P}}_{k-1}\mathbf{F}_{k-1}^T + \mathbf{L}_{k-1}\mathbf{Q}\mathbf{L}_{k-1}^T$
3. **Error Kalman Update:** $\delta\mathbf{y}_k = \mathbf{y}_k - \mathbf{h}(\check{\mathbf{x}}_k)$, $\mathbf{K}_k = \check{\mathbf{P}}_k\mathbf{H}_k^T(\mathbf{H}_k\check{\mathbf{P}}_k\mathbf{H}_k^T + \mathbf{R})^{-1}$
4. **Injection & Reset:** $\hat{\mathbf{x}}_k = \check{\mathbf{x}}_k \oplus \delta\hat{\mathbf{x}}_k$, $\delta\hat{\mathbf{x}}_k \leftarrow \mathbf{0}$, $\hat{\mathbf{P}}_k = (\mathbf{I} - \mathbf{K}_k\mathbf{H}_k)\check{\mathbf{P}}_k$


---
## 3. Implementing the `ErrorStateKalmanFilter` Class

### 📝 Exercise 1: Implement the ES-EKF Class


In [ ]:
def wrap_angle(angle):
    """Wraps angle to [-pi, pi]."""
    return (angle + np.pi) % (2 * np.pi) - np.pi

class ErrorStateKalmanFilter:
    """2D Error-State Extended Kalman Filter (ES-EKF)."""
    
    def __init__(self, x0: np.ndarray, P0: np.ndarray):
        self.x_nom = np.asarray(x0, dtype=np.float64).reshape(4, 1)
        self.P = np.asarray(P0, dtype=np.float64)
        
    def propagate(self, u: np.ndarray, dt: float, Q: np.ndarray):
        px, py, v, theta = self.x_nom.flatten()
        a, omega = u.flatten()
        
        # Nominal state integration
        self.x_nom[0, 0] = px + v * np.cos(theta) * dt
        self.x_nom[1, 0] = py + v * np.sin(theta) * dt
        self.x_nom[2, 0] = v + a * dt
        self.x_nom[3, 0] = wrap_angle(theta + omega * dt)
        
        # Error state Jacobian F
        F = np.array([
            [1.0, 0.0, np.cos(theta) * dt, -v * np.sin(theta) * dt],
            [0.0, 1.0, np.sin(theta) * dt,  v * np.cos(theta) * dt],
            [0.0, 0.0, 1.0,                 0.0],
            [0.0, 0.0, 0.0,                 1.0]
        ])
        L = np.eye(4)
        self.P = F @ self.P @ F.T + L @ Q @ L.T
        
    def update_gps(self, y_gps: np.ndarray, R_gps: np.ndarray):
        y_vec = np.asarray(y_gps, dtype=np.float64).reshape(2, 1)
        h_x = self.x_nom[:2, :]
        delta_y = y_vec - h_x
        
        H = np.array([
            [1.0, 0.0, 0.0, 0.0],
            [0.0, 1.0, 0.0, 0.0]
        ])
        S = H @ self.P @ H.T + R_gps
        K = self.P @ H.T @ np.linalg.inv(S)
        delta_x = K @ delta_y
        
        # Injection into nominal state
        self.x_nom += delta_x
        self.x_nom[3, 0] = wrap_angle(self.x_nom[3, 0])
        
        # Reset error covariance
        I = np.eye(4)
        self.P = (I - K @ H) @ self.P
        self.P = 0.5 * (self.P + self.P.T)
        return delta_y, S

print('ES-EKF class compiled successfully.')


---
## 4. Physical Formulation of IMU Noise $\mathbf{Q}_{\text{imu}}$ & GPS Noise $\mathbf{R}_{\text{gps}}$

### 🔹 IMU Process Noise $\mathbf{Q}_{\text{imu}}$ (100 Hz, $\Delta t = 0.01\text{ s}$):
In strapdown navigation, high-frequency IMU sensor noise is modeled as continuous white noise spectral density, scaled by step time $\Delta t$:
$$\mathbf{Q}_k \approx \mathbf{Q}_{\text{continuous}} \cdot \Delta t_{\text{imu}}$$
* $\sigma_v = 0.05\text{ m/s}/\sqrt{\text{s}}$: Accelerometer Velocity Random Walk (VRW).
* $\sigma_\theta = 0.01\text{ rad/s}/\sqrt{\text{s}}$: Gyroscope Angular Random Walk (ARW).
$$\mathbf{Q}_{\text{imu}} = \operatorname{diag}(0.01^2, 0.01^2, 0.05^2, 0.01^2) \times 0.01$$

### 🔹 GPS Measurement Noise $\mathbf{R}_{\text{gps}}$ (10 Hz):
* $\sigma_{p_x} = 1.5\text{ m}, \sigma_{p_y} = 1.5\text{ m}$: Standard civilian single-frequency GPS pseudorange precision.
$$\mathbf{R}_{\text{gps}} = \operatorname{diag}(1.5^2, 1.5^2)$$


---
## 5. Simulation: Dead-Reckoning with 10s GPS Outage


In [ ]:
dt_imu = 0.01   # 100 Hz IMU
T_total = 50.0
N_steps = int(T_total / dt_imu)
time = np.linspace(0, T_total, N_steps)

Q_imu = np.diag([0.01**2, 0.01**2, 0.05**2, 0.01**2]) * dt_imu
R_gps = np.diag([1.5**2, 1.5**2])

x_true = np.array([0.0, 0.0, 10.0, 0.0]).reshape(4, 1)
x_true_all = np.zeros((4, N_steps))

u_cmds = []
for k in range(N_steps):
    t = time[k]
    a = 0.2 * np.sin(0.1 * t)
    omega = 0.05 * np.cos(0.05 * t)
    u_cmds.append(np.array([a, omega]))

x0_est = np.array([0.0, 0.0, 10.0, 0.0]).reshape(4, 1)
P0_est = np.diag([1.0**2, 1.0**2, 1.0**2, np.deg2rad(5.0)**2])
esekf = ErrorStateKalmanFilter(x0_est, P0_est)

x_est_all = np.zeros((4, N_steps))
P_diag_all = np.zeros((4, N_steps))
gps_meas_history = []

for k in range(N_steps):
    t = time[k]
    u = u_cmds[k].reshape(2, 1)
    w = np.random.multivariate_normal(np.zeros(4), Q_imu).reshape(4, 1)
    px, py, v, theta = x_true.flatten()
    x_true[0, 0] += v * np.cos(theta) * dt_imu
    x_true[1, 0] += v * np.sin(theta) * dt_imu
    x_true[2, 0] += u[0, 0] * dt_imu
    x_true[3, 0] = wrap_angle(x_true[3, 0] + u[1, 0] * dt_imu)
    x_true += w
    x_true_all[:, k] = x_true.flatten()

    noisy_u = u + np.random.normal(0, [0.05, 0.01]).reshape(2, 1)
    esekf.propagate(noisy_u, dt_imu, Q_imu)

    is_gps_step = (k % 10 == 0)
    is_outage = (20.0 <= t <= 30.0)
    if is_gps_step and not is_outage:
        v_gps = np.random.multivariate_normal(np.zeros(2), R_gps).reshape(2, 1)
        y_gps = x_true[:2, :] + v_gps
        esekf.update_gps(y_gps, R_gps)
        gps_meas_history.append((t, y_gps[0, 0], y_gps[1, 0]))

    x_est_all[:, k] = esekf.x_nom.flatten()
    P_diag_all[:, k] = np.diag(esekf.P)

print('ES-EKF simulation completed.')


---
## 6. Plotly Interactive Outage Diagnostics


In [ ]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        '2D Trajectory Fusion (GPS + IMU)',
        'Position Error vs. 3-Sigma (GPS Outage Highlighted)',
        'Velocity Error vs. 3-Sigma Bound',
        'Heading Error vs. 3-Sigma Bound'
    )
)

gps_arr = np.array(gps_meas_history)
fig.add_trace(go.Scatter(x=x_true_all[0, :], y=x_true_all[1, :], mode='lines', name='Ground Truth', line=dict(color='black', width=3)), row=1, col=1)
fig.add_trace(go.Scatter(x=gps_arr[:, 1], y=gps_arr[:, 2], mode='markers', name='GPS Fixes (10 Hz)', marker=dict(color='red', size=4, opacity=0.6)), row=1, col=1)
fig.add_trace(go.Scatter(x=x_est_all[0, :], y=x_est_all[1, :], mode='lines', name='ES-EKF Estimate', line=dict(color='blue', width=2, dash='dash')), row=1, col=1)

pos_err = np.sqrt((x_true_all[0, :] - x_est_all[0, :])**2 + (x_true_all[1, :] - x_est_all[1, :])**2)
sigma_pos = 3.0 * np.sqrt(P_diag_all[0, :] + P_diag_all[1, :])
fig.add_trace(go.Scatter(x=time, y=pos_err, mode='lines', name='Position Error [m]', line=dict(color='blue', width=1.5)), row=1, col=2)
fig.add_trace(go.Scatter(x=time, y=sigma_pos, mode='lines', name='3-Sigma Bound [m]', line=dict(color='red', dash='dot', width=1.5)), row=1, col=2)

v_err = np.abs(x_true_all[2, :] - x_est_all[2, :])
sigma_v = 3.0 * np.sqrt(P_diag_all[2, :])
fig.add_trace(go.Scatter(x=time, y=v_err, mode='lines', name='Speed Error [m/s]', line=dict(color='green', width=1.5)), row=2, col=1)
fig.add_trace(go.Scatter(x=time, y=sigma_v, mode='lines', name='3-Sigma Bound [m/s]', line=dict(color='red', dash='dot', width=1.5)), row=2, col=1)

th_err = np.abs(wrap_angle(x_true_all[3, :] - x_est_all[3, :]))
sigma_th = 3.0 * np.sqrt(P_diag_all[3, :])
fig.add_trace(go.Scatter(x=time, y=np.rad2deg(th_err), mode='lines', name='Heading Error [deg]', line=dict(color='magenta', width=1.5)), row=2, col=2)
fig.add_trace(go.Scatter(x=time, y=np.rad2deg(sigma_th), mode='lines', name='3-Sigma Bound [deg]', line=dict(color='red', dash='dot', width=1.5)), row=2, col=2)

# Add shaded GPS outage box on Position Error
fig.add_vrect(x0=20.0, x1=30.0, fillcolor='yellow', opacity=0.25, line_width=0, annotation_text='GPS Outage Window', annotation_position='top left', row=1, col=2)

fig.update_layout(title_text='Day 2 ES-EKF: Dead-Reckoning & GNSS/INS Fusion', template='plotly_white', height=750, width=1050)
fig.show()


---
## 7. Summary
The ES-EKF isolates high-frequency nonlinear kinematics in the nominal state while using linear estimation for small perturbation errors, delivering superior numerical stability.
